In [ ]:
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from typing import List, Tuple, Dict, TypedDict

In [ ]:
initial_year = 2010
final_year = 2035
export_plot_data = True

In [ ]:
output_path = "f0_OSMOSYS_ECU_Output.csv"
planmicc_path = "docs/PLANMICCOutput.csv"
ndc_path = "docs/NDCOutput.csv" 

df = pd.read_csv(output_path)
planmicc = pd.read_csv(planmicc_path)
ndc = pd.read_csv(ndc_path)

In [ ]:
# Filter only co2e emissions 
def get_model_emissions(dataFrame: pd.DataFrame, emissions: List[str]) -> pd.DataFrame:
    emission_filters = (dataFrame['Year']<=final_year) & (~dataFrame['Emission'].isna()) & (~dataFrame['AnnualTechnologyEmission'].isna()) & (dataFrame['Emission'].isin(emissions))
    emissions_output = dataFrame.loc[emission_filters, ['Strategy', 'Emission','Year', 'AnnualTechnologyEmission']]
    missing_emissions =  set(emissions) - set(emissions_output['Emission'].unique())
    if len(missing_emissions) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_emissions)
    emissions_output['AnnualTechnologyEmission'] = emissions_output['AnnualTechnologyEmission'] * 1000
    energy_emissions = emissions_output.pivot_table(index = ['Year'], columns='Strategy' ,values = 'AnnualTechnologyEmission', aggfunc='sum').reset_index()
    return energy_emissions

In [ ]:
def draw_model_emission_scatter(name:str, dataFrame:pd.DataFrame) -> List[go.Scatter]:
    data = []
    for scenario in dataFrame.columns[1:]:
        data.append(go.Scatter(name=f"{scenario}({name})", x=dataFrame['Year'], y=dataFrame[scenario]))
    
    return data

# Fig. Emisiones Totales
def draw_emissions_trayectory(emissions:List[Tuple[str,pd.DataFrame]], title:str):
    fig_data=[]
    for model in emissions:
        fig_data += draw_model_emission_scatter(model[0], model[1])
    
    fig = go.Figure(data=fig_data)


    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":title,
        },
        title_x=0.5,
        yaxis_title = "[GgCO2e]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """)

    fig.show()

In [ ]:
ndc_emissions = get_model_emissions(ndc, [
    'CO2e_FUG', 
    'CO2e_PP',
    'CO2e_REF', 
    'CO2e_DE'
    ]) 
planmicc_emissions= get_model_emissions(planmicc, ["CO2"])
df_emissions = get_model_emissions(df, ["CO2"])

if export_plot_data :
    df_emissions.to_csv('Exports/current_emissions.csv', index=False)

draw_emissions_trayectory([
    ('Current', df_emissions), 
    ('PLANMICC', planmicc_emissions), 
    ('NDC', ndc_emissions)
],
title="Trayectoria Emisiones sector Energía"
)

In [ ]:
# Emissions by technology

# Dictionary for technology grouping [GroupName]:[Technologies]
grouped_emissions = {
  "Agricultura/Silvicultura/Pesca": ["T5LPGAGR", "T5PURGSLAGR"],
  "Aviación Civil": ["T5KJFAERTRN", "T5PURGSLAERTRN"],
  "Comercial/Institucional": [
    "T5FOICOM",
    "T5LPGCOM",
    "T5PURDSLCOM",
    "T5PURGSLCOM"
  ],
  "Generación de electricidad": [
    "Diésel MCI",
    "Diésel Turbo Gas",
    "Fuel Oil MCI",
    "Fuel Oil Turbina de Vapor",
    "Gas Natural Turbo Gas",
    "PP_WASICE"
  ],
  "Industrias manufactureras": [
    "T5FIRIND",
    "T5FOIIND",
    "T5LPGIND",
    "T5NGSIND",
    "T5PURDSLIND",
    "T5PURGSLIND"
  ],
  "Navegación marítima y fluvial": [
    "T5FOISHITRN",
    "T5PURDSLSHITRN",
    "T5PURGSLSHITRN"
  ],
  "No especificado": ["T5KJFCON", "T5LPGCON", "T5PURDSLCON", "T5PURGSLCON"],
  "Otras industrias de la energía": [
    "PPICRUPETICE",
    "PPICRUPETTST",
    "PPIDSLELRICE",
    "PPIDSLGALICE",
    "PPIDSLPETICE",
    "PPIDSLPETTGS",
    "PPINGSPETICE",
    "PPINGSPETTGS",
    "PPINGSPETTST"
  ],
  "Otro tipo de transporte": ["T5CRUCRUTRN"],
  "Refinación del petróleo": ["REFCRU"],
  "Residencial": ["T5FIRRES", "T5LPGRES", "T5NGSRES"],
  "Transporte terrestre": [
    "T4_DSLHEA",
    "T4_DSLLIG",
    "T4_DSLMED",
    "T4_DSLPRI",
    "T4_DSLPUB",
    "T4_GSLLIG",
    "T4_GSLPRI",
    "T4_GSLPUB",
    "T4_LPGHEA",
    "T4_NGSMED",
    "T4_NGSPUB"
  ]
}

def get_energy_tech_emissions_by_strategy(strategy:str):
  value_groups = {x: k for k, v in grouped_emissions.items() for x in v}
  technology_emission_filters = (df['Year']<=final_year) &(~df['Emission'].isna()) & (~df['AnnualTechnologyEmission'].isna()) & (df['Emission'] == "CO2") & (df['Strategy'] == strategy)
  technology_emissions_output = df.loc[technology_emission_filters, ['Technology', 'Emission','Year', 'AnnualTechnologyEmission']]
  technology_emissions_output['Technology'] = technology_emissions_output['Technology'].replace(value_groups) # replace tech codes for group codes
  technology_emissions_output['AnnualTechnologyEmission'] = technology_emissions_output['AnnualTechnologyEmission'].apply(lambda x : x * 1000) # transform megatonns to gigagrams
  energy_technology_emissions = technology_emissions_output.pivot_table(index = ['Year'], columns='Technology' ,values = 'AnnualTechnologyEmission', aggfunc='sum').reset_index()
  return energy_technology_emissions


def draw_area_tech_emissions_chart(strategy:str):
    energy_technology_emissions = get_energy_tech_emissions_by_strategy(strategy)
    # Fig. Emissions per Technology vs Year
    fig = px.area(energy_technology_emissions, 
                x="Year", 
                y=energy_technology_emissions.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": "Emisiones [Gg CO2 eq]"
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Detalle de emisiones por Tecnología en el sector Energía - {strategy}",
        },
        title_x=0.5
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f}</b>
    """)
    fig.show()

In [ ]:
scenarios = list(df['Strategy'].unique())
for scenario in scenarios:
    draw_area_tech_emissions_chart(scenario)

In [ ]:
# installed capacity
grouped_installed_capacity = {
  "Biogás": ["PP_BGSICE"],
  "Biomasa": ["PP_BMSTST", "PP_SUG"],
  "Eólica": ["PP_WND_US"],
  "Geotérmica": ["PP_GEO"],
  "Hidro": [
    "PP_HYDAMADAMLAR",
    "PP_HYDAMADAMMED",
    "PP_HYDAMARORLAR",
    "PP_HYDAMARORMED",
    "PP_HYDAMARORSMA",
    "PP_HYDPACDAMMED",
    "PP_HYDPACDAMSMA",
    "PP_HYDPACRORLAR",
    "PP_HYDPACRORMED",
    "PP_HYDPACRORSMA"
  ],
  "Residuos MCI": ["PP_WASICE"],
  "Solar": ["PP_SPV_US"],
  "Térmicas Fósiles": [
    "Diésel MCI",
    "Diésel Turbo Gas",
    "Fuel Oil MCI",
    "Fuel Oil Turbina de Vapor",
    "Gas Natural Turbo Gas"
  ]
}

def sum_and_multiply_by_2(series):
   return series.sum() * 1000

def get_energy_installed_capacity_by_strategy(strategy:str):
  capacity_groups = {x: k for k, v in grouped_installed_capacity.items() for x in v}
  installed_capacity_filters =  (df['Year']<=final_year) &(~df['Technology'].isna()) & (~df['TotalCapacityAnnual'].isna()) & (df['Strategy'] == strategy)
  installed_capacity_output = df.loc[installed_capacity_filters, ['Technology','Year', 'TotalCapacityAnnual']]
  #installed_capacity_output['Technology'] = installed_capacity_output['Technology'].replace(capacity_groups) # replace tech codes for group codes, ignore if not mapped
  installed_capacity_output['Technology'] = installed_capacity_output['Technology'].map(capacity_groups) # replace tech codes for group codes, ignore if not mapped
  installed_capacity = installed_capacity_output.pivot_table(index = ['Year'], columns='Technology' ,values = 'TotalCapacityAnnual', aggfunc=sum_and_multiply_by_2).reset_index()
  return installed_capacity


def draw_area_installed_capacity_chart(strategy:str):
    energy_installed_capacity = get_energy_installed_capacity_by_strategy(strategy)
    # Fig. Emissions per Technology vs Year
    fig = px.area(energy_installed_capacity, 
                x="Year", 
                y=energy_installed_capacity.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": "Capacidad[MW]"
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Capacidad Instalada en el sector Energía - {strategy}",
        },
        title_x=0.5
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Capacidad:</span> <b>%{y:.3f}</b>
    """)
    fig.show()


In [ ]:
for scenario in scenarios:
    draw_area_installed_capacity_chart(scenario)

In [ ]:
def draw_area_installed_capacity_percentage_chart(strategy:str):
    energy_installed_capacity = get_energy_installed_capacity_by_strategy(strategy)

    # Fig. Emissions per Technology vs Year
    fig = px.area(energy_installed_capacity, 
                x="Year", 
                y=energy_installed_capacity.columns[1:],
                groupnorm='percent',
                labels={
                    "variable":"",
                    "Year": "Año",
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Porcentaje de Capacidad Instalada en el sector Energía - {strategy}",
        },
        title_x=0.5,
        yaxis_title = "Participación de la capacidad<br>instalada por tipo de planta [%]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Porcentaje:</span> <b>%{y:.3f} %</b>
    """)
    fig.show()

In [ ]:
for scenario in scenarios:
    draw_area_installed_capacity_percentage_chart(scenario)

In [ ]:
# Generated Energy
grouped_generation = {
  "Biogás": ["PP_BGSICE"],
  "Biomasa": ["PP_BMSTST", "PP_SUG"],
  "Eólica": ["PP_WND_US"],
  "Geotérmica": ["PP_GEO"],
  "Hidro": [
    "PP_HYDAMADAMLAR",
    "PP_HYDAMADAMMED",
    "PP_HYDAMARORLAR",
    "PP_HYDAMARORMED",
    "PP_HYDAMARORSMA",
    "PP_HYDPACDAMMED",
    "PP_HYDPACDAMSMA",
    "PP_HYDPACRORLAR",
    "PP_HYDPACRORMED",
    "PP_HYDPACRORSMA"
  ],
  "Residuos MCI": ["PP_WASICE"],
  "Solar": ["PP_SPV_US"],
  "Térmicas Fósiles": [
    "Diésel MCI",
    "Diésel Turbo Gas",
    "Fuel Oil MCI",
    "Fuel Oil Turbina de Vapor",
    "Gas Natural Turbo Gas"
  ]
}

def sum_and_division_for_gen(series):
   return series.sum() / 0.0036

def get_generated_energy_by_strategy(strategy:str):
  generation_groups = {x: k for k, v in grouped_generation.items() for x in v}
  generated_energy_filters = (df['Year']<=final_year) & (~df['Technology'].isna()) & (~df['TotalTechnologyAnnualActivity'].isna()) & (df['Strategy'] == strategy)
  generated_energy_output = df.loc[generated_energy_filters, ['Technology','Year', 'TotalTechnologyAnnualActivity']]
  generated_energy_output['Technology'] = generated_energy_output['Technology'].map(generation_groups) # replace tech codes for group codes, ignore if not mapped
  generated_energy = generated_energy_output.pivot_table(index = ['Year'], columns='Technology' ,values = 'TotalTechnologyAnnualActivity', aggfunc=sum_and_division_for_gen).reset_index()
  return generated_energy

def draw_area_installed_capacity_percentage_chart(strategy:str):
    energy_installed_capacity = get_energy_installed_capacity_by_strategy(strategy)

    # Fig. Emissions per Technology vs Year
    fig = px.area(energy_installed_capacity, 
                x="Year", 
                y=energy_installed_capacity.columns[1:],
                groupnorm='percent',
                labels={
                    "variable":"",
                    "Year": "Año",
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Porcentaje de Capacidad Instalada en el sector Energía - {strategy}",
        },
        title_x=0.5,
        yaxis_title = "Participación de la capacidad<br>instalada por tipo de planta [%]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Emisiones:</span> <b>%{y:.3f} %</b>
    """)
    fig.show()

def draw_area_generated_energy_chart(strategy:str):
    energy_generated = get_generated_energy_by_strategy(strategy)
    # Fig. Emissions per Technology vs Year
    fig = px.area(energy_generated, 
                x="Year", 
                y=energy_generated.columns[1:], 
                labels={
                    "variable":"",
                    "Year": "Año",
                    "value": "[GWh]"
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Energía generada por tipo de fuente en el sector Energía - {strategy}",
        },
        title_x=0.5
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Energía Generada:</span> <b>%{y:.3f}</b>
    """)
    fig.show()


In [ ]:
for scenario in scenarios:
    draw_area_generated_energy_chart(scenario)

In [ ]:
def get_produced_energy_by_strategy(strategy:str):
    production_groups = {x: k for k, v in grouped_generation.items() for x in v}
    produced_energy_filters = (df['Year']<=final_year) & (~df['Technology'].isna()) & (~df['ProductionByTechnology'].isna()) & (df['Strategy'] == strategy)
    produced_energy_output = df.loc[produced_energy_filters, ['Technology','Year', 'ProductionByTechnology']]
    produced_energy_output['Technology'] = produced_energy_output['Technology'].map(production_groups) # replace tech codes for group codes, ignore if not mapped
    totalProductionByTech = produced_energy_output['ProductionByTechnology'].sum()
    produced_energy = produced_energy_output.pivot_table(index = ['Year'], columns='Technology' ,values = 'ProductionByTechnology', aggfunc=lambda x: x.sum()/totalProductionByTech).reset_index()
    return produced_energy

def draw_area_produced_energy_percentage_chart(strategy:str):
    produced_energy = get_produced_energy_by_strategy(strategy)

    # Fig. Emissions per Technology vs Year
    fig = px.area(produced_energy, 
                x="Year", 
                y=produced_energy.columns[1:],
                groupnorm='percent',
                labels={
                    "variable":"",
                    "Year": "Año",
                },
                )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Generación agrupada en el sector Energía - {strategy}",
        },
        title_x=0.5,
        yaxis_title = "Generación[%]"
    )

    fig.update_traces(hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Porcentaje:</span> <b>%{y:.3f} %</b>
    """)
    fig.show()

In [ ]:
for scenario in scenarios:
    draw_area_produced_energy_percentage_chart(scenario)

In [ ]:
import plotly.graph_objects as go

def get_energy_costs():
    produced_energy_output = df.loc[df['Year']<=final_year,['Strategy', 'Externalities2023', 'Capex2023', 'FixedOpex2023', 'VarOpex2023']]
    produced_energy_output.loc[:,'Total'] = produced_energy_output[['Externalities2023', 'Capex2023', 'FixedOpex2023', 'VarOpex2023']].sum(axis=1)
    bar = produced_energy_output.groupby(['Strategy']).sum().agg(lambda x:x/1000).reset_index()
    return bar

def draw_energy_costs_histogram():
    energy_costs = get_energy_costs()

    # Fig. Emissions per Technology vs Year
    fig = go.Figure(data=[
        go.Bar(name="Capitales", x=energy_costs['Strategy'], y=energy_costs['Capex2023'], text=energy_costs['Capex2023'], textposition='inside', texttemplate="%{y:.3f}", insidetextanchor="middle"),
        go.Bar(name="Operación fijos", x=energy_costs['Strategy'], y=energy_costs['FixedOpex2023'], text=energy_costs['FixedOpex2023'], textposition='inside', texttemplate="%{y:.3f}",insidetextanchor="middle"),
        go.Bar(name="Operación variable",x=energy_costs['Strategy'], y=energy_costs['VarOpex2023'], text=energy_costs['VarOpex2023'], textposition='inside', texttemplate="%{y:.3f}",insidetextanchor="middle"),
        go.Bar(name="Externalidades",x=energy_costs['Strategy'], y=energy_costs['Externalities2023'], text=energy_costs['Externalities2023'], textposition='inside', texttemplate="%{y:.3f}",insidetextanchor="middle"),
        ],
        layout=go.Layout(barmode='stack')
        )

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor='lightgrey',
    )

    fig.update_layout(
        plot_bgcolor="white",
        title = {
            "text":f"Impacto Economico sector Energía",
        },
        title_x=0.5,
        yaxis_title = "Costos [miles de millones de US$]"
    )

    fig.update_traces(hovertemplate="""
    <span>Escenario:</span> <b>%{x}</b>
    <span>Costo:</span> <b>%{y:.3f}</b>
    """)

    fig.add_trace(go.Scatter(mode='text',
                         x=energy_costs['Strategy'],
                         y=energy_costs['Total'],
                         textposition='top center',
                         text=[str(x) for x in energy_costs['Total'].tolist()],
                         texttemplate="%{y:.3f}",
                         showlegend=False,
                         hoverinfo='skip'
                        ))
    
    fig.show()

draw_energy_costs_histogram()

In [ ]:
grouped_demand_fuels = {
    "propane": ["E5_PROGAS"],
    "sugarcane": ["E5_INDSUG"],
    "firewood": ["E5_INDFIR", "E5_RESFIR"],
    # "exportation": [
    #     "E5_EXPCRU",
    #     "E5_EXPELE",
    #     "E5_EXPFOI",
    #     "E5_EXPPURDSL",
    #     "E5_EXPPURGSL",
    # ],
    "crudeoil": ["E5_CONREF", "E5_CRUTRNCRU"],
    "fueloil": ["E5_COMFOI", "E5_INDFOI", "E5_SHITRNFOI"],
    "noenergetic": ["E5_AGRNOE", "E5_CONNOE"],
    "keronsenejet": ["E5_AERTRNKJF", "E5_CONKJF"],
    "naturalgas": [
        "E4_NGSLIG",
        "E4_NGSMED",
        "E4_NGSPRI",
        "E4_NGSPUB",
        "E5_INDNGS",
        "E5_RESNGS",
    ],
    "glp": [
        "E4_LPGHEA",
        "E4_LPGLIG",
        "E4_LPGPRI",
        "E4_LPGPUB",
        "E5_AGRLPG",
        "E5_COMLPG",
        "E5_CONLPG",
        "E5_INDLPG",
        "E5_RESLPG",
    ],
    "hidrogen": ["E4_HYDHEA", "E4_HYDPUB", "E5_INDHYD"],
    "gasoil": [
        "E4_GSLHEA",
        "E4_GSLLIG",
        "E4_GSLPRI",
        "E4_GSLPUB",
        "E5_AERTRNPURGSL",
        "E5_AGRPURGSL",
        "E5_COMPURGSL",
        "E5_CONPURGSL",
        "E5_INDPURGSL",
        "E5_SHITRNPURGSL",
    ],
    "electricity": [
        "E4_ELEHEA",
        "E4_ELELIG",
        "E4_ELEMED",
        "E4_ELEPRI",
        "E4_ELEPUB",
        "E5_AGRELE",
        "E5_COMELE",
        "E5_CONELE",
        "E5_CRUTRNELE",
        "E5_INDELE",
        "E5_RESELE",
        "E5_TRNFRERAI",
        "E5_TRNPASRAI",
    ],
    "diesel": [
        "E4_DSLHEA",
        "E4_DSLLIG",
        "E4_DSLMED",
        "E4_DSLPRI",
        "E4_DSLPUB",
        "E5_COMPURDSL",
        "E5_CONPURDSL",
        "E5_INDPURDSL",
        "E5_SHITRNPURDSL",
    ],
    # "other": [
    #     "DEMTRN_NOMOT",
    #     "DEMTRNFREHEA",
    #     "DEMTRNFRELIG",
    #     "DEMTRNFREMED",
    #     "DEMTRNPASPRI",
    #     "DEMTRNPASPUB",
    #     "E0_BGS",
    #     "E0_BMS",
    #     "E0_CRU",
    #     "E0_FIR",
    #     "E0_NGS",
    #     "E0_SUG",
    #     "E0_WAS",
    #     "E1_BIODSL",
    #     "E1_COK",
    #     "E1_CRU",
    #     "E1_DSL",
    #     "E1_ELE",
    #     "E1_ETA",
    #     "E1_FOI",
    #     "E1_GAS",
    #     "E1_GSL",
    #     "E1_HEC",
    #     "E1_KJF",
    #     "E1_LPG",
    #     "E1_NGS",
    #     "E1_NOE",
    #     "E1_PURDSL",
    #     "E1_PURGSL",
    #     "E1_REDCRU",
    #     "E2_ELE",
    #     "E2_HYD",
    #     "E2_NGS",
    #     "E3_ELE",
    #     "E3_HYD",
    #     "E5_CRUTRN",
    #     "E5_EXPREDCRU",
    #     "E5_TRNBUS",
    #     "E5_TRNCAM",
    #     "E5_TRNFREHEA",
    #     "E5_TRNFRELIG",
    #     "E5_TRNFREMED",
    #     "E5_TRNMIC",
    #     "E5_TRNMOT",
    #     "E5_TRNSED",
    #     "E5_TRNSUV",
    #     "E5_TRNTAX",
    # ],
}

grouped_demand_fuels_labels = {
    "propane": "Propano",
    "sugarcane": "Caña de Azucar",
    "firewood": "Leña",
    "exportation": "Exportaciones",
    "crudeoil": "Petróleo Crudo",
    "fueloil": "Fuel Oil",
    "noenergetic": "No Energético",
    "keronsenejet": "Keroseno y Jet Fuel",
    "naturalgas": "Gas Natural",
    "glp": "GLP",
    "hidrogen": "Hidrógeno",
    "gasoil": "Gasolina",
    "electricity": "Electricidad",
    "diesel": "Diésel",
    "other": "Otros",
}

# SUM([ProductionByTechnology])/.005807


def get_fuels_demmand_by_strategy(
    model_df: pd.DataFrame, fuels: Dict[str, List[str]], strategy: str
):
    demmand_filters = (
        (model_df["Year"] >= initial_year)
        & (model_df["Year"] <= final_year)
        & (~model_df["Fuel"].isna())
        & (~model_df["ProductionByTechnology"].isna())
        & (model_df["Fuel"].isin(fuels))
        & (model_df["Strategy"] == strategy)
    )

    demmand_output = model_df.loc[
        demmand_filters, ["Fuel", "Year", "ProductionByTechnology"]
    ]

    missing_fuels = set(fuels) - set(demmand_output["Fuel"].unique())

    if len(missing_fuels) > 0:
        print("WARNING: The following emissions are missing:")
        print(missing_fuels)

    demmand_df = (
        demmand_output.groupby(["Year"])["ProductionByTechnology"]
        .apply(lambda x: x.sum() / 0.005807)
        .reset_index()
    )

    return demmand_df


def draw_grouped_fuels_demmand_area_by_strategy(
    model_df: pd.DataFrame,
    grouped_fuels: Dict[str, List[str]],
    grouped_fuels_labels: Dict[str, str],
    strategy: str,
    title: str,
):
    fig_data = []
    for k, fuels in grouped_fuels.items():
        demmand_df = get_fuels_demmand_by_strategy(model_df, fuels, strategy)
        fig_data.append(
            go.Scatter(
                x=demmand_df["Year"],
                y=demmand_df["ProductionByTechnology"],
                mode="lines",
                stackgroup="one",
                name=grouped_fuels_labels[k],
            )
        )

    fig = go.Figure(data=fig_data)

    fig.update_yaxes(
        mirror=True,
        showline=False,
        gridcolor="lightgrey",
    )

    fig.update_layout(
        plot_bgcolor="white",
        title={
            "text": title,
        },
        title_x=0.5,
        yaxis_title="Área [Mha]",
    )

    fig.update_traces(
        hovertemplate="""
    <span>Año:</span> <b>%{x}</b>
    <span>Valor:</span> <b>%{y:.3f}</b>
    """
    )

    fig.update_layout(
        legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="right", x=1)
    )

    fig.update_layout(yaxis=dict(nticks=10))

    fig.show()

def draw_model_grouped_fuels_demands_area(model_df:pd.DataFrame, grouped_fuels:Dict[str,List[str]], grouped_fuels_labels:Dict[str, str]):
    for strategy in model_df['Strategy'].unique():
        draw_grouped_fuels_demmand_area_by_strategy(model_df, grouped_fuels, grouped_fuels_labels, strategy, f"Demanda de energía por fuente - {strategy}")

In [ ]:
draw_model_grouped_fuels_demands_area(df, grouped_demand_fuels, grouped_demand_fuels_labels)